# Face Recognition Evaluation (TAR @ FAR)

In this notebook, we will evaluate the biometric performance of our FaceNet (InceptionResnetV1) model on our dataset.
According to the project PRD, we need to output an evaluation report plotting the ROC curve and calculating the True Accept Rate (TAR) at various False Accept Rates (FAR).

We will do this by:
1. Extracting 512-D embeddings for all faces in our dataset.
2. Generating all possible pairs of faces (Positive Pairs = same person, Negative Pairs = different people).
3. Calculating the Euclidean distance for every pair.
4. Using these distances to compute the ROC curve, AUC, and TAR@FAR.

In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from facenet_pytorch import MTCNN, InceptionResnetV1
from pathlib import Path
from itertools import combinations
import sys

# Add the parent directory to sys.path so we can import our dataset loader
sys.path.append(str(Path.cwd().parent))
from dataset import get_dataloader

## 1. Extract Embeddings

In [ ]:
# Initialize device and models
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

mtcnn = MTCNN(image_size=160, margin=0, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# Load dataset
loader, classes = get_dataloader(batch_size=16)

embeddings = []
labels = []

print("Extracting embeddings for evaluation...")
for images, batch_labels in loader:
    for i, img in enumerate(images):
        face = mtcnn(img)
        if face is not None:
            face = face.unsqueeze(0).to(device)
            with torch.no_grad():
                emb = resnet(face).squeeze().cpu().numpy()
            
            embeddings.append(emb)
            labels.append(batch_labels[i])

embeddings = np.array(embeddings)
labels = np.array(labels)
print(f"Successfully extracted {len(embeddings)} embeddings.")

Using device: cpu
Extracting embeddings for evaluation...


AttributeError: 'int' object has no attribute 'item'

## 2. Generate Face Pairs and Calculate Distances

In [8]:
# Generate all possible combinations of pairs
indices = range(len(labels))
pairs = list(combinations(indices, 2))

distances = []
is_same = []

for i, j in pairs:
    # Calculate L2 (Euclidean) distance between the two embeddings
    dist = np.linalg.norm(embeddings[i] - embeddings[j])
    distances.append(dist)
    
    # 1 if they are the same person, 0 if different
    is_same.append(1 if labels[i] == labels[j] else 0)

distances = np.array(distances)
is_same = np.array(is_same)

print(f"Total pairs generated: {len(pairs)}")
print(f"Positive pairs (Same Person): {sum(is_same)}")
print(f"Negative pairs (Different People): {len(is_same) - sum(is_same)}")

Total pairs generated: 0
Positive pairs (Same Person): 0
Negative pairs (Different People): 0


## 3. Plot ROC Curve and Calculate TAR@FAR

In [9]:
# roc_curve expects higher scores to indicate a positive match.
# Since a smaller distance means a higher likelihood of a match, we use negative distance.
scores = -distances

fpr, tpr, thresholds = roc_curve(is_same, scores)
roc_auc = auc(fpr, tpr)

def get_tar_at_far(target_far, fpr, tpr):
    """Finds the True Accept Rate (TAR) at a specific False Accept Rate (FAR)"""
    # Find the index where FAR is closest to the target_far without exceeding it significantly
    idx = np.where(fpr <= target_far)[0][-1]
    return tpr[idx]

# Calculate TAR at typical benchmark FARs
tar_at_1_percent = get_tar_at_far(0.01, fpr, tpr)
tar_at_10_percent = get_tar_at_far(0.10, fpr, tpr)

print("--- EVALUATION RESULTS ---")
print(f"Area Under Curve (AUC): {roc_auc:.4f}")
print(f"TAR @ FAR=1% : {tar_at_1_percent:.4f}")
print(f"TAR @ FAR=10%: {tar_at_10_percent:.4f}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Accept Rate (FAR)')
plt.ylabel('True Accept Rate (TAR)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

ValueError: operands could not be broadcast together with shapes (0,) (2,) 